<a href="https://colab.research.google.com/github/Levan-Danelia/FRTB/blob/main/FRTB_CRVG_Non_Securitization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Setup and Initial Portfolio (Steps 1 & 2)
# -----------------------------------------------------------------------------
# This cell loads the initial portfolio data and regulatory parameters.
# The output displays the starting positions and their gross sensitivities, which
# corresponds to the tables in Step 1 and Step 2 of the report.

# Import necessary libraries
import pandas as pd
import numpy as np

# --- Initial Portfolio Data ---
# This data represents the starting positions and their gross vega sensitivities.
portfolio_data = [
    {'position_id': 1, 'bucket': 5, 'issuer': 'Issuer 1', 'tenor': '5Y', 'gross_sensitivity': 269},
    {'position_id': 2, 'bucket': 5, 'issuer': 'Issuer 1', 'tenor': '5Y', 'gross_sensitivity': 309},
    {'position_id': 3, 'bucket': 5, 'issuer': 'Issuer 2', 'tenor': '5Y', 'gross_sensitivity': -302},
    {'position_id': 4, 'bucket': 5, 'issuer': 'Issuer 2', 'tenor': '5Y', 'gross_sensitivity': -44},
]

# --- Regulatory Parameters ---
VEGA_RISK_WEIGHT = 1.00  # 100% for Vega CSR non-securitisation
RHO_DELTA_DIFFERENT_ISSUER_IG = 0.35 # 35% for different issuers in same IG bucket

# Create the initial DataFrame
df = pd.DataFrame(portfolio_data)
df['risk_factor'] = df['issuer'] + ' - ' + df['tenor'] + ' Volatility'

# --- Output for Step 1 ---
print("--- Step 1: Definition of Risk Factors ---")
print("The calculation begins by identifying the unique risk factor for each position.")
print("\n" + df[['position_id', 'bucket', 'issuer', 'tenor', 'risk_factor']].to_string(index=False))

# --- Output for Step 2 ---
print("\n\n--- Step 2: Definition of Vega Sensitivities ---")
print("The gross sensitivity for each position is the starting point for the calculation.")
print("\n" + df[['position_id', 'risk_factor', 'gross_sensitivity']].to_string(index=False))

--- Step 1: Definition of Risk Factors ---
The calculation begins by identifying the unique risk factor for each position.

 position_id  bucket   issuer tenor              risk_factor
           1       5 Issuer 1    5Y Issuer 1 - 5Y Volatility
           2       5 Issuer 1    5Y Issuer 1 - 5Y Volatility
           3       5 Issuer 2    5Y Issuer 2 - 5Y Volatility
           4       5 Issuer 2    5Y Issuer 2 - 5Y Volatility


--- Step 2: Definition of Vega Sensitivities ---
The gross sensitivity for each position is the starting point for the calculation.

 position_id              risk_factor  gross_sensitivity
           1 Issuer 1 - 5Y Volatility                269
           2 Issuer 1 - 5Y Volatility                309
           3 Issuer 2 - 5Y Volatility               -302
           4 Issuer 2 - 5Y Volatility                -44


In [ ]:
# Cell 2: Net Sensitivities (Step 3)
# -----------------------------------------------------------------------------
# As per Article 325f(5), sensitivities for identical risk factors are netted.
# We group by the risk factor and sum the sensitivities.

df_net = df.groupby(['risk_factor']).agg(
    net_sensitivity=('gross_sensitivity', 'sum')
).reset_index()

print("\n\n--- Step 3: Net Sensitivities ---")
print("Gross sensitivities for each unique risk factor are summed to get the net sensitivity.")
print("\n" + df_net.to_string(index=False))



--- Step 3: Net Sensitivities ---
Gross sensitivities for each unique risk factor are summed to get the net sensitivity.

             risk_factor  net_sensitivity
Issuer 1 - 5Y Volatility              578
Issuer 2 - 5Y Volatility             -346


In [ ]:
# Cell 3: Weighted Sensitivities (Step 4)
# -----------------------------------------------------------------------------
# Each net sensitivity is multiplied by the regulatory risk weight (100%).

df_weighted = df_net.copy()
df_weighted['risk_weight'] = f"{VEGA_RISK_WEIGHT:.0%}"
df_weighted['weighted_sensitivity'] = df_weighted['net_sensitivity'] * VEGA_RISK_WEIGHT

# Calculate Sb for later steps
S_b = df_weighted['weighted_sensitivity'].sum()

print("\n\n--- Step 4: Risk Weights & Weighted Sensitivities ---")
print("Net sensitivities are multiplied by the regulatory risk weight for the bucket.")
print("\n" + df_weighted[['risk_factor', 'net_sensitivity', 'risk_weight', 'weighted_sensitivity']].to_string(index=False))
print(f"\nSum of Weighted Sensitivities (S₅): {S_b:,.2f}")



--- Step 4: Risk Weights & Weighted Sensitivities ---
Net sensitivities are multiplied by the regulatory risk weight for the bucket.

             risk_factor  net_sensitivity risk_weight  weighted_sensitivity
Issuer 1 - 5Y Volatility              578        100%                 578.0
Issuer 2 - 5Y Volatility             -346        100%                -346.0

Sum of Weighted Sensitivities (S₅): 232.00


In [ ]:
# Cell 4: Intra-Bucket Correlation (Step 5)
# -----------------------------------------------------------------------------
# Per Article 325ay, the Vega correlation is derived from the delta correlation.
# For different issuers in the same IG bucket (Bucket 5) and identical option
# maturities, the correlation is 35%.

rho_kl_medium = RHO_DELTA_DIFFERENT_ISSUER_IG

print("\n\n--- Step 5: Intra-Bucket Correlation ---")
print("The correlation between the two risk factors in Bucket 5 is determined.")
print("Formula: ρₖₗ = min( ρₖₗ(DELTA) × ρₖₗ(option maturity), 1 )")
print(f"ρₖₗ(DELTA) for different IG issuers in same sector = {RHO_DELTA_DIFFERENT_ISSUER_IG:.0%}")
print("ρₖₗ(option maturity) for identical tenors = 100%")
print("----------------------------------------------------------")
print(f"Medium Scenario Correlation (ρₖₗ): {rho_kl_medium:.2%}")



--- Step 5: Intra-Bucket Correlation ---
The correlation between the two risk factors in Bucket 5 is determined.
Formula: ρₖₗ = min( ρₖₗ(DELTA) × ρₖₗ(option maturity), 1 )
ρₖₗ(DELTA) for different IG issuers in same sector = 35%
ρₖₗ(option maturity) for identical tenors = 100%
----------------------------------------------------------
Medium Scenario Correlation (ρₖₗ): 35.00%


In [ ]:
# Cell 5: Intra-Bucket Aggregation - Medium Scenario (Step 7)
# -----------------------------------------------------------------------------
# Calculate the bucket-specific capital charge (K_b) for the Medium Scenario.

ws_values = df_weighted['weighted_sensitivity'].tolist()
ws1 = ws_values[0]
ws2 = ws_values[1]

sum_ws_sq = ws1**2 + ws2**2
cross_term = 2 * rho_kl_medium * ws1 * ws2

K_b_medium = np.sqrt(max(0, sum_ws_sq + cross_term))

print("\n\n--- Step 7: Intra-Bucket Aggregation (Medium Scenario) ---")
print("Weighted sensitivities are aggregated using the specified correlation.")
print("\n1. Sum of Squares (Σ WSₖ²):")
print(f"   ({ws1:.2f}² + {ws2:.2f}²) = {sum_ws_sq:,.2f}")
print("\n2. Sum of Cross-Products (2 * ρₖₗ * WS₁ * WS₂):")
print(f"   (2 * {rho_kl_medium:.2f} * {ws1:.2f} * {ws2:.2f}) = {cross_term:,.2f}")
print("\n3. Bucket 5 Capital (K₅):")
print(f"   √({sum_ws_sq:,.2f} + {cross_term:,.2f}) = √({(sum_ws_sq + cross_term):,.2f}) = {K_b_medium:,.2f}")



--- Step 7: Intra-Bucket Aggregation (Medium Scenario) ---
Weighted sensitivities are aggregated using the specified correlation.

1. Sum of Squares (Σ WSₖ²):
   (578.00² + -346.00²) = 453,800.00

2. Sum of Cross-Products (2 * ρₖₗ * WS₁ * WS₂):
   (2 * 0.35 * 578.00 * -346.00) = -139,991.60

3. Bucket 5 Capital (K₅):
   √(453,800.00 + -139,991.60) = √(313,808.40) = 560.19


In [ ]:
# Cell 6: Correlation Scenarios (Step 9)
# -----------------------------------------------------------------------------
# Recalculate the total capital for High and Low correlation scenarios.

def calculate_capital_for_scenario(correlation):
    """Helper function to calculate K_b for a given correlation."""
    sum_ws_sq = ws1**2 + ws2**2
    cross_term = 2 * correlation * ws1 * ws2
    return np.sqrt(max(0, sum_ws_sq + cross_term))

# Calculate high and low correlations
rho_kl_high = min(rho_kl_medium * 1.25, 1.0)
rho_kl_low = max(2 * rho_kl_medium - 1.0, 0.75 * rho_kl_medium)

# Calculate capital for each scenario
K_b_high = calculate_capital_for_scenario(rho_kl_high)
K_b_low = calculate_capital_for_scenario(rho_kl_low)

scenario_data = {
    'Scenario': ['Medium Correlation', 'High Correlation', 'Low Correlation'],
    'Intra-Bucket Correlation': [f"{rho_kl_medium:.2%}", f"{rho_kl_high:.2%}", f"{rho_kl_low:.2%}"],
    'Capital Requirement': [K_b_medium, K_b_high, K_b_low]
}
df_scenarios = pd.DataFrame(scenario_data)

print("\n\n--- Step 9: Correlation Scenarios ---")
print("The capital is recalculated under stressed correlation assumptions.")
# Format the capital requirement column
df_scenarios['Capital Requirement'] = df_scenarios['Capital Requirement'].apply(lambda x: f"{x:,.2f}")
print("\n" + df_scenarios.to_string(index=False))



--- Step 9: Correlation Scenarios ---
The capital is recalculated under stressed correlation assumptions.

          Scenario Intra-Bucket Correlation Capital Requirement
Medium Correlation                   35.00%              560.19
  High Correlation                   43.75%              528.03
   Low Correlation                   26.25%              590.60


In [ ]:
# Cell 7: Final Charge Calculation (Step 10)
# -----------------------------------------------------------------------------
# The final charge is the maximum of the three scenarios.

final_charge = max(K_b_medium, K_b_high, K_b_low)
winning_scenario = "Medium"
if final_charge == K_b_high:
    winning_scenario = "High"
elif final_charge == K_b_low:
    winning_scenario = "Low"


print("\n\n--- Step 10: Final Charge Calculation ---")
print("The final requirement is the maximum of the three scenarios.")
print("----------------------------------------------------------")
print(f" Final Credit Vega Capital Requirement: {final_charge:,.2f}")
print(f" (Driven by the {winning_scenario} Correlation Scenario)")
print("----------------------------------------------------------")



--- Step 10: Final Charge Calculation ---
The final requirement is the maximum of the three scenarios.
----------------------------------------------------------
 Final Credit Vega Capital Requirement: 590.60
 (Driven by the Low Correlation Scenario)
----------------------------------------------------------
